# Lipid compound and lipid droplet analysis

## Imports


In [ ]:
%matplotlib inline
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import glob
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import pandas as pd 

import czifile
from cellpose import models
from skimage import exposure
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops
from skimage.morphology import remove_small_objects, white_tophat, disk, closing
from skimage.segmentation import relabel_sequential, watershed
from scipy.ndimage import distance_transform_edt, binary_fill_holes



## Parameters


In [ ]:

BASE_DIR = "."  
IMAGE_FOLDER_NAMES = [
    "52726 images",
    "60326 images",
    "60826 images",
    "61026 images",
    "61526 images",
    "61726 images",
    "62426 images",
]
IMAGE_FOLDERS = [os.path.join(BASE_DIR, name) for name in IMAGE_FOLDER_NAMES]

FILE_EXTENSION = "*.czi"
RESULTS_FOLDER = "Results"
MASK_FOLDER = os.path.join(RESULTS_FOLDER, "masks")

SHOW_PLOTS = True  # set False to skip the inline 6-panel figures on long batch runs

USE_GPU = True

# Whole-cell segmentation (cyto3 on orange + blue)
CYTO_DIAMETER = 80
CYTO_FLOW_THRESH = 0.4
CYTO_PROB_THRESH = -15.0
CELL_MIN_AREA = 300

# Nucleus segmentation (nuclei model on blue)
NUC_DIAMETER= 100
NUC_FLOW_THRESH = 1
NUC_PROB_THRESH = -20.0
NUC_MIN_AREA = 100

# Hybrid cell fill: recover droplet-packed cells cyto3 misses, seeded from nuclei
CELL_FILL_FG_THRESH = 0.06 # normalized-orange cutoff marking cell foreground to grow into
CELL_GROW_DIST = 70

# Lipid droplet segmentation (white top-hat on the raw orange channel)
LIPID_TOPHAT_RADIUS = 15 # structuring-element radius (px); set just above the largest droplet
LIPID_MIN_SIZE = 6 

os.makedirs(RESULTS_FOLDER, exist_ok=True)
os.makedirs(MASK_FOLDER, exist_ok=True)


def date_tag_from_folder(folder):
    return os.path.basename(os.path.normpath(folder)).split()[0]


def mask_folder_for(date_tag):
    path = os.path.join(MASK_FOLDER, date_tag)
    os.makedirs(path, exist_ok=True)
    return path


cyto_model = models.Cellpose(gpu=USE_GPU, model_type='cyto3')
nuc_model = models.Cellpose(gpu=USE_GPU, model_type='nuclei')
print(f"cyto3 + nuclei ready, GPU={USE_GPU}")

## Helper functions


In [ ]:
# Display and loading helpers
def normalize_image(image):
    """Min-max normalize to float64 [0, 1] (for segmentation + display, never for MFI)."""
    img = image.astype(np.float64)
    mn, mx = img.min(), img.max()
    return np.zeros_like(img) if mx == mn else (img - mn) / (mx - mn)


def boost_contrast_visual(image):
    """Percentile stretch for DISPLAY ONLY (never used for measurements)."""
    p_lo, p_hi = np.percentile(image, (1, 99.5))
    return exposure.rescale_intensity(image, in_range=(p_lo, p_hi))


def load_channels(filepath):
    """Load a .czi and return the raw (orange, blue) channels exactly as stored."""
    try:
        img = np.squeeze(czifile.CziFile(filepath).asarray())
    except Exception as e:
        print(f"  [ERROR] {e}")
        return None, None

    if img.ndim == 3 and img.shape[0] in (2, 3, 4):
        ch0, ch1 = img[0], img[1]
    elif img.ndim == 3 and img.shape[-1] in (2, 3, 4):
        ch0, ch1 = img[..., 0], img[..., 1]
    else:
        print(f"  [ERROR] Unexpected shape: {img.shape}")
        return None, None

    orange_raw, blue_raw = ch0, ch1 # swap here if panels show orange/blue flipped
    return orange_raw, blue_raw


def drop_small_labels(labels, min_area):
    """Zero out labeled regions smaller than min_area, then relabel 1..N."""
    labels = labels.astype(np.int32)
    counts = np.bincount(labels.ravel())
    small = np.flatnonzero(counts < min_area)
    small = small[small != 0] # keep background (label 0)
    if small.size:
        labels[np.isin(labels, small)] = 0
    return relabel_sequential(labels)[0]


# CELL MASK — cyto3 on orange + blue 
def segment_whole_cells(orange_norm, blue_norm):
    """Cellpose cyto3 on the 2-channel stack. channels=[1, 2]: orange=cytoplasm, blue=nucleus."""
    img_2ch = np.stack([
        (orange_norm * 255).astype(np.uint8), # -> channel "1": cytoplasm
        (blue_norm   * 255).astype(np.uint8), # -> channel "2": nucleus
    ], axis=-1)

    masks, _, _, _ = cyto_model.eval(
        img_2ch, diameter=CYTO_DIAMETER, channels=[1, 2],
        flow_threshold=CYTO_FLOW_THRESH, cellprob_threshold=CYTO_PROB_THRESH,
        do_3D=False, resample=True,
    )
    cells = drop_small_labels(masks, CELL_MIN_AREA)
    print(f"    cyto3 cells: {cells.max()}")
    return cells


# NUCLEUS MASK — nuclei model on blue only 
def segment_nuclei(blue_norm):
    """Cellpose nuclei model on the blue channel (grayscale, channels=[0, 0])."""
    masks, _, _, _ = nuc_model.eval(
        (blue_norm * 255).astype(np.uint8), diameter=NUC_DIAMETER, channels=[0, 0],
        flow_threshold=NUC_FLOW_THRESH, cellprob_threshold=NUC_PROB_THRESH,
        do_3D=False, resample=True,
    )
    nuclei = drop_small_labels(masks, NUC_MIN_AREA)
    print(f"    nuclei:      {nuclei.max()}")
    return nuclei


# HYBRID FILL — add cells for nuclei that cyto3 missed 
def fill_missed_cells_from_nuclei(cyto_cells, nuclei, orange_norm):
    cells = cyto_cells.astype(np.int32).copy()
    n_cyto = int(cells.max())

    # seed map = existing cells + one fresh label per missed nucleus
    seeds = cells.copy()
    next_label = n_cyto + 1
    for nid in np.unique(nuclei):
        if nid == 0:
            continue
        nuc = nuclei == nid
        if (cells[nuc] > 0).mean() < 0.5: # nucleus not covered by a cell
            seeds[nuc & (cells == 0)] = next_label
            next_label += 1
    n_added = next_label - 1 - n_cyto
    if n_added == 0:
        print(f"    hybrid cells: {n_cyto} (+0)")
        return cells

    # grow the new seeds outward, bounded by real signal and a max distance
    dist_from_seed = distance_transform_edt(seeds == 0)
    foreground = orange_norm > CELL_FILL_FG_THRESH
    foreground = binary_fill_holes(closing(foreground, disk(3))) | (seeds > 0)
    foreground &= dist_from_seed <= CELL_GROW_DIST
    grown = watershed(dist_from_seed, markers=seeds, mask=foreground)

    new_pixels = (grown > n_cyto) & (cyto_cells == 0)  # Never overwrite Cellpose cells
    cells[new_pixels] = grown[new_pixels]
    cells = drop_small_labels(cells, CELL_MIN_AREA)
    print(f"    hybrid cells: {cells.max()} (+{n_added} recovered from nuclei)")
    return cells


# LIPID MASK — white top-hat on the RAW orange channel 
def segment_lipid_droplets(orange_raw, labeled_cells):
    """
    Detect lipid droplets as bright puncta inside cells, independent of how bright
    each image is overall. 
    """
    in_cells = labeled_cells > 0
    contrast = white_tophat(orange_raw, disk(LIPID_TOPHAT_RADIUS)).astype(np.float64)
    vals = contrast[in_cells & (contrast > 0)]
    thresh = threshold_otsu(vals) if vals.size else np.inf
    lipid_mask = remove_small_objects((contrast > thresh) & in_cells, min_size=LIPID_MIN_SIZE)
    print(f"    lipid px:    {lipid_mask.sum():,}")
    return lipid_mask


## Image segmentation


In [ ]:
cmap_orange = LinearSegmentedColormap.from_list("O", [(0, 0, 0), (1, 0.55, 0)])
cmap_blue   = LinearSegmentedColormap.from_list("B", [(0, 0, 0), (0.3, 0.6, 1)])


def segment_one_image(filepath, mask_dir):
    """Segment one .czi, save its 4 masks into mask_dir, and (optionally) visualize."""
    filename = os.path.basename(filepath)
    stem = os.path.splitext(filename)[0]
    print(f"Processing: {filename}")

    orange_raw, blue_raw = load_channels(filepath)
    if orange_raw is None:
        print("  Skipping.\n")
        return

    orange_norm = normalize_image(orange_raw)
    blue_norm   = normalize_image(blue_raw)

    # 1. cyto3 cells   2. nuclei   2b. fill cyto3-missed cells from nuclei
    cyto3_cells    = segment_whole_cells(orange_norm, blue_norm)
    labeled_nuclei = segment_nuclei(blue_norm)
    labeled_cells  = fill_missed_cells_from_nuclei(cyto3_cells, labeled_nuclei, orange_norm)

    if labeled_cells.max() == 0:
        print("  No cells found, skipping.\n")
        return

    # 3. lipid droplets — top-hat on the RAW orange channel, restricted to cells
    lipid_mask = segment_lipid_droplets(orange_raw, labeled_cells)

    # 4. cytoplasm = cell − nucleus
    cyto_mask = (labeled_cells > 0) & ~(labeled_nuclei > 0)

    # Save each mask into this folder's own mask subdir
    np.save(os.path.join(mask_dir, f"{stem}_cell.npy"),    labeled_cells)
    np.save(os.path.join(mask_dir, f"{stem}_nucleus.npy"), labeled_nuclei)
    np.save(os.path.join(mask_dir, f"{stem}_cyto.npy"),    cyto_mask)
    np.save(os.path.join(mask_dir, f"{stem}_lipid.npy"),   lipid_mask)

    if SHOW_PLOTS:
        n_filled = int(labeled_cells.max()) - int(cyto3_cells.max())

        # Visualize the 6 panels
        fig, axes = plt.subplots(1, 6, figsize=(34, 6))
        fig.patch.set_facecolor("black")
        fig.suptitle(filename, color="white", fontsize=9, y=1.01)

        axes[0].imshow(boost_contrast_visual(orange_raw), cmap=cmap_orange)
        axes[0].set_title("Orange (Lipids — raw)", color="white", fontsize=9)

        axes[1].imshow(boost_contrast_visual(blue_raw), cmap=cmap_blue)
        axes[1].set_title("Blue (Nucleus — raw)", color="white", fontsize=9)

        axes[2].imshow(np.where(labeled_cells > 0, labeled_cells, np.nan),
                       cmap="nipy_spectral", interpolation="none", vmin=1)
        axes[2].set_title(f"Cell Mask (cyto3 {cyto3_cells.max()} + filled {n_filled})",
                          color="white", fontsize=9)

        axes[3].imshow(np.where(labeled_nuclei > 0, labeled_nuclei, np.nan),
                       cmap="nipy_spectral", interpolation="none", vmin=1)
        axes[3].set_title(f"Nucleus Mask ({labeled_nuclei.max()})", color="white", fontsize=9)

        axes[4].imshow(cyto_mask.astype(np.uint8), cmap="gray", vmin=0, vmax=1)
        axes[4].set_title("Cytoplasm (cell − nucleus)", color="white", fontsize=9)

        # lipid mask overlaid on the raw orange so capture quality is visible at a glance
        axes[5].imshow(boost_contrast_visual(orange_raw), cmap="gray")
        axes[5].imshow(np.ma.masked_where(~lipid_mask, lipid_mask), cmap="autumn", vmin=0, vmax=1)
        axes[5].set_title(f"Lipid Mask — top-hat ({lipid_mask.sum():,} px)", color="white", fontsize=9)

        for ax in axes:
            ax.axis("off")
            ax.set_facecolor("black")

        plt.tight_layout()
        plt.show()

    print(f"  Saved 4 masks to {mask_dir}\n")


def segment_folder(image_folder):
    """Segment every .czi in one folder, saving masks to Results/masks/<date>/."""
    date_tag = date_tag_from_folder(image_folder)
    mask_dir = mask_folder_for(date_tag)
    image_files = sorted(glob.glob(os.path.join(image_folder, FILE_EXTENSION)))
    print(f"{date_tag}: segmenting {len(image_files)} file(s)\n")
    for filepath in image_files:
        segment_one_image(filepath, mask_dir)


## Sequential processing and per-cell analysis

For each folder: segment images, save masks, analyze cells, and write one CSV.


In [ ]:
def analyze_file(filepath, mask_dir):
    """Return a list of per-cell measurement dicts for one image (masks read from mask_dir)."""
    filename = os.path.basename(filepath)
    stem = os.path.splitext(filename)[0]

    # Re-load raw orange channel for intensity (no segmentation is re-run)
    orange_raw, blue_raw = load_channels(filepath)
    if orange_raw is None:
        return []

    # STRICTLY RAW: cast uint16 -> float64 (values identical, just safe to average).
    orange_f = orange_raw.astype(np.float64)

    # Overall brightness of THIS raw image (reference for the relative MFI below).
    image_mean_raw = float(orange_f.mean())

    def mean_raw(sel):
        """Mean of the RAW orange pixels under a boolean mask (NaN if mask is empty)."""
        return float(orange_f[sel].mean()) if sel.any() else np.nan

    def rel(mfi):
        """Express a raw MFI relative to how bright the raw image is overall."""
        return (mfi / image_mean_raw) if image_mean_raw > 0 else np.nan

    # Load the masks the segmentation step already saved (this folder's mask subdir)
    try:
        labeled_cells = np.load(os.path.join(mask_dir, f"{stem}_cell.npy"))
        labeled_nuclei = np.load(os.path.join(mask_dir, f"{stem}_nucleus.npy"))
        cyto_mask = np.load(os.path.join(mask_dir, f"{stem}_cyto.npy")).astype(bool)
        lipid_mask = np.load(os.path.join(mask_dir, f"{stem}_lipid.npy")).astype(bool)
    except FileNotFoundError:
        print(f"  [skip] masks not found for {filename} — run the segmentation step first.")
        return []

    rows = []
    cell_ids = np.unique(labeled_cells)
    cell_ids = cell_ids[cell_ids != 0]# drop background (0)

    for cid in cell_ids:
        cell_mask = labeled_cells == cid

        # Restrict each saved mask to THIS cell
        cyto_cell = cyto_mask & cell_mask # cytoplasm mask, this cell
        lipid_cell = lipid_mask & cell_mask #lipid mask, this cell
        cyto_no_lipid = cyto_cell & ~lipid_mask # cytoplasm minus lipid mask

        #Surface areas (pixel cnts)
        cyto_area = int(cyto_cell.sum()) # SA of cytoplasm  (cyto mask)
        lipid_area = int(lipid_cell.sum())# SA of lipid droplets (lipid mask)
        sa_ratio = (lipid_area / cyto_area) if cyto_area > 0 else np.nan #RATIO COLUMN LIPID SA / CYTO SA

        # Mean fluorescence intensity on the RAW orange channel (strictly raw)
        mfi_lipid = mean_raw(lipid_cell) # lipid mask
        mfi_cyto = mean_raw(cyto_cell)  # cyto mask
        mfi_cyto_no_lipid = mean_raw(cyto_no_lipid) # cyto mask AND NOT lipid mask

        rows.append({
            "Filename" : filename, 
            "Cell_ID" : int(cid),
            "Cell_Area_px" : int(cell_mask.sum()),
            "Nucleus_Area_px" : int(((labeled_nuclei > 0) & cell_mask).sum()),
            "Cytoplasm_Surface_Area_Without_Nucleus_including_LDS_px" : cyto_area, # cyto mask
            "Lipid_Droplet_Surface_Area_px" : lipid_area, # lipid mask
            "Image_Mean_Orange_raw" : image_mean_raw, # whole-image raw brightness (reference)
            "MFI_Lipid_Droplets" : mfi_lipid, # raw, lipid mask
            "MFI_Cytoplasm_all_inclusive" : mfi_cyto, # raw, cyto mask
            "MFI_Cytoplasm_no_Lipid_Droplets" : mfi_cyto_no_lipid, # raw, cyto mask AND NOT lipid mask
            "MFI_Lipid_Droplets_rel_image" : rel(mfi_lipid), # raw MFI / image brightness
            "MFI_Cytoplasm_all_inclusive_rel_image" : rel(mfi_cyto), # raw MFI / image brightness
            "MFI_Cytoplasm_no_Lipid_Droplets_rel_image" : rel(mfi_cyto_no_lipid), # raw MFI / image brightness
            "SA_LD/SA_Cytoplasm_without_nucleus" : sa_ratio, #ratio between sa of lipid droplets and the sa of cyto no nucleus
        })

    return rows


def analyze_folder(image_folder):
    """Measure every image in one folder and write its own {date}_..._analysis.csv."""
    date_tag = date_tag_from_folder(image_folder)
    mask_dir = mask_folder_for(date_tag)
    output_csv = os.path.join(RESULTS_FOLDER, f"{date_tag}_lipid_compound_lipid_droplet_analysis.csv")

    image_files = sorted(glob.glob(os.path.join(image_folder, FILE_EXTENSION)))
    print(f"{date_tag}: analyzing {len(image_files)} file(s)")

    all_rows = []
    for filepath in image_files:
        fname = os.path.basename(filepath)
        print(f"Analyzing: {fname}")
        rows = analyze_file(filepath, mask_dir)
        print(f"  {len(rows)} cells measured")
        all_rows.extend(rows)

    df = pd.DataFrame(all_rows)
    df.to_csv(output_csv, index=False)
    print(f"Saved {len(df)} cell rows -> {output_csv}")
    return df



last_df = None
for image_folder in IMAGE_FOLDERS:
    date_tag = date_tag_from_folder(image_folder)
    if not os.path.isdir(image_folder):
        print(f" {date_tag}: folder '{image_folder}' not found (skip)\n")
        continue

    print(f"{date_tag}: START ('{image_folder}')")
    segment_folder(image_folder) 
    last_df = analyze_folder(image_folder)
    print(f"{date_tag}:\n")

print("All folders processed — one CSV each in Results/.")
last_df.head() if last_df is not None else None